# ML-3: Random Forest Model Training & SMOTE Experimentation

This notebook trains and evaluates Random Forest failure prediction models on the full AI4I 2020 Predictive Maintenance Dataset (~10,000 rows).

## Key Design & Implementation Rules:
1. **Full Dataset Usage**: Uses the full dataset (`data/cleaned_predictive_maintenance.csv`). No small subsets.
2. **Stratified 80/20 Train/Test Split**: Performs one stratified split. The 20% test set is kept **completely untouched** for ML-4.
3. **Evaluated Approaches**:
   - **Approach A**: Balanced Random Forest (`class_weight='balanced'`)
   - **Approach B**: SMOTE + Random Forest (`SMOTE` inside `imblearn.pipeline`)
4. **Strict Leakage-Free SMOTE Rule**: SMOTE is applied **strictly inside each training fold** during cross-validation via `imblearn.pipeline.Pipeline`. SMOTE is **never** applied before splitting or to test/validation sets.
5. **5-Fold Stratified Cross-Validation**: Models are evaluated on the 80% training set across Recall, Precision, F1, PR-AUC, and ROC-AUC.
6. **Model Selection & Artifact Export**: The winning approach based on CV metrics (emphasizing PR-AUC and F1-Score) is fitted on the complete 80% training set and exported to `ml/models/best_model.joblib` for downstream ML-4 evaluation.

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Imbalanced-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

sys.path.append(os.path.abspath("../src"))

from feature_engineering import (
    DomainFeatureEngineer,
    build_preprocessing_pipeline,
    EXCLUDED_COLS,
    TARGET_COL,
    BASE_FEATURE_COLS,
    CATEGORICAL_COLS,
    ALL_NUMERICAL_COLS
)

## 1. Ingest Full Cleaned Dataset (~10,000 rows)

We load the full cleaned dataset and exclude leakage-prone failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) and identifiers (`UDI`, `Product ID`).

In [2]:
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)
print(f"Loaded full dataset shape: {df.shape}")

feature_cols = [c for c in df.columns if c not in EXCLUDED_COLS and c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

print(f"Predictive Input Features ({len(feature_cols)}): {feature_cols}")
print(f"Target ({TARGET_COL}) Class Distribution:")
print(y.value_counts(normalize=True))

Loaded full dataset shape: (10000, 7)
Predictive Input Features (6): ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target (Machine failure) Class Distribution:
Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64


## 2. Stratified 80/20 Train/Test Split (Untouched Test Set)

We perform a single stratified 80/20 train/test split. The 20% test set is preserved completely untouched for final evaluation in ML-4.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape} (Failures: {y_train.sum()})")
print(f"X_test shape:  {X_test.shape} (Failures: {y_test.sum()} - Untouched for ML-4)")

X_train shape: (8000, 6) (Failures: 271)
X_test shape:  (2000, 6) (Failures: 68 - Untouched for ML-4)


## 3. Define Candidate Pipelines & Leakage-Free SMOTE Design

We construct two candidate pipelines using `imblearn.pipeline.Pipeline`:
1. **Approach A (Balanced Random Forest)**: Uses `class_weight='balanced'` in Random Forest.
2. **Approach B (SMOTE + Random Forest)**: Embeds `SMOTE` inside the pipeline **after** domain engineering and preprocessing, ensuring SMOTE runs strictly on training folds.

In [4]:
# ============================================================
# 4 EXPERIMENTS
# ============================================================

def build_experiment_preprocessor(scaler):
    """
    One-Hot encode Type and scale numerical features.
    The scaler is supplied separately for each experiment.
    """

    return ColumnTransformer(
        transformers=[
            (
                'cat',
                OneHotEncoder(
                    sparse_output=False,
                    handle_unknown='ignore'
                ),
                CATEGORICAL_COLS
            ),
            (
                'num',
                scaler,
                ALL_NUMERICAL_COLS
            )
        ],
        remainder='passthrough',
        verbose_feature_names_out=False
    )


def build_rf_pipeline(scaler, use_smote):
    """
    Creates one complete leakage-safe experiment pipeline.

    Order:
        Domain Feature Engineering
                ↓
        One-Hot + Scaling
                ↓
        Optional SMOTE
                ↓
        Random Forest
    """

    steps = [
        ('engineer', DomainFeatureEngineer()),
        ('preprocessor', build_experiment_preprocessor(scaler))
    ]

    if use_smote:
        steps.append(
            ('smote', SMOTE(random_state=42))
        )

    steps.append(
        (
            'classifier',
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1,
                class_weight=None if use_smote else 'balanced'
            )
        )
    )

    return ImbPipeline(steps)


# ------------------------------------------------------------
# Experiment 1
# One-Hot + StandardScaler + Balanced Random Forest
# ------------------------------------------------------------

pipeline_1 = build_rf_pipeline(
    scaler=StandardScaler(),
    use_smote=False
)


# ------------------------------------------------------------
# Experiment 2
# One-Hot + MinMaxScaler + Balanced Random Forest
# ------------------------------------------------------------

pipeline_2 = build_rf_pipeline(
    scaler=MinMaxScaler(),
    use_smote=False
)


# ------------------------------------------------------------
# Experiment 3
# One-Hot + StandardScaler + SMOTE + Random Forest
# ------------------------------------------------------------

pipeline_3 = build_rf_pipeline(
    scaler=StandardScaler(),
    use_smote=True
)


# ------------------------------------------------------------
# Experiment 4
# One-Hot + MinMaxScaler + SMOTE + Random Forest
# ------------------------------------------------------------

pipeline_4 = build_rf_pipeline(
    scaler=MinMaxScaler(),
    use_smote=True
)


experiment_pipelines = {
    "OneHot + StandardScaler + Balanced RF": pipeline_1,
    "OneHot + MinMaxScaler + Balanced RF": pipeline_2,
    "OneHot + StandardScaler + SMOTE + RF": pipeline_3,
    "OneHot + MinMaxScaler + SMOTE + RF": pipeline_4
}

print("Created 4 experiment pipelines:")
for name in experiment_pipelines:
    print(f"  • {name}")

Created 4 experiment pipelines:
  • OneHot + StandardScaler + Balanced RF
  • OneHot + MinMaxScaler + Balanced RF
  • OneHot + StandardScaler + SMOTE + RF
  • OneHot + MinMaxScaler + SMOTE + RF


## 4. 5-Fold Stratified Cross-Validation on Training Set

We evaluate both candidate approaches using **5-Fold Stratified K-Fold CV** strictly on the training set.

In [5]:
# ============================================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'pr_auc': 'average_precision',
    'roc_auc': 'roc_auc'
}

cv_comparison = []

for name, pipeline in experiment_pipelines.items():

    print(f"\nRunning CV for: {name}")

    cv_res = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=skf,
        scoring=scoring,
        n_jobs=-1
    )

    cv_comparison.append({
        "Experiment": name,
        "CV Recall": round(np.mean(cv_res['test_recall']), 4),
        "CV Precision": round(np.mean(cv_res['test_precision']), 4),
        "CV F1": round(np.mean(cv_res['test_f1']), 4),
        "CV PR-AUC": round(np.mean(cv_res['test_pr_auc']), 4),
        "CV ROC-AUC": round(np.mean(cv_res['test_roc_auc']), 4)
    })


comp_df = pd.DataFrame(cv_comparison)

print("\n" + "=" * 80)
print("4-EXPERIMENT CROSS-VALIDATION COMPARISON")
print("=" * 80)

print(
    comp_df.to_string(index=False)
)


Running CV for: OneHot + StandardScaler + Balanced RF

Running CV for: OneHot + MinMaxScaler + Balanced RF

Running CV for: OneHot + StandardScaler + SMOTE + RF

Running CV for: OneHot + MinMaxScaler + SMOTE + RF

4-EXPERIMENT CROSS-VALIDATION COMPARISON
                           Experiment  CV Recall  CV Precision  CV F1  CV PR-AUC  CV ROC-AUC
OneHot + StandardScaler + Balanced RF     0.8119        0.9238 0.8640     0.8872      0.9754
  OneHot + MinMaxScaler + Balanced RF     0.8119        0.9163 0.8606     0.8869      0.9754
 OneHot + StandardScaler + SMOTE + RF     0.8266        0.7197 0.7684     0.8592      0.9770
   OneHot + MinMaxScaler + SMOTE + RF     0.8376        0.6310 0.7193     0.8459      0.9762


## 5. Model Selection & Complete Training Set Fitting

We select the winning pipeline based on CV metrics (emphasizing PR-AUC, F1, and Recall), fit it on the full 80% training set, and export the trained artifact to `ml/models/best_model.joblib`.

In [6]:
# ============================================================
# SELECT BEST EXPERIMENT
# ============================================================

# Rank primarily by PR-AUC, then F1, then Recall.
# These are more informative than accuracy for the imbalanced
# failure-prediction problem.

sorted_df = comp_df.sort_values(
    by=['CV PR-AUC', 'CV F1', 'CV Recall'],
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("EXPERIMENT RANKING")
print("=" * 80)

print(sorted_df.to_string(index=False))

winning_experiment = sorted_df.iloc[0]["Experiment"]

print("\n" + "=" * 80)
print(f"CURRENT WINNER: {winning_experiment}")
print("=" * 80)

print(
    "\nNOTE: The current winner is selected using CV results only. "
    "The untouched test set has NOT been used."
)

EXPERIMENT RANKING
                           Experiment  CV Recall  CV Precision  CV F1  CV PR-AUC  CV ROC-AUC
OneHot + StandardScaler + Balanced RF     0.8119        0.9238 0.8640     0.8872      0.9754
  OneHot + MinMaxScaler + Balanced RF     0.8119        0.9163 0.8606     0.8869      0.9754
 OneHot + StandardScaler + SMOTE + RF     0.8266        0.7197 0.7684     0.8592      0.9770
   OneHot + MinMaxScaler + SMOTE + RF     0.8376        0.6310 0.7193     0.8459      0.9762

CURRENT WINNER: OneHot + StandardScaler + Balanced RF

NOTE: The current winner is selected using CV results only. The untouched test set has NOT been used.


In [7]:
# ============================================================
# SAVE ML-3 EXPERIMENT RESULTS
# ============================================================

models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

results_path = os.path.join(
    models_dir,
    "ml3_four_experiment_comparison.json"
)

with open(results_path, "w") as f:
    json.dump(
        {
            "experiments": cv_comparison,
            "ranking": sorted_df.to_dict(orient="records"),
            "cv_folds": 5,
            "random_state": 42,
            "selection_priority": [
                "CV PR-AUC",
                "CV F1",
                "CV Recall"
            ]
        },
        f,
        indent=2
    )

print(
    f"Saved experiment comparison to:\n"
    f"{os.path.abspath(results_path)}"
)

Saved experiment comparison to:
c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\ml3_four_experiment_comparison.json


## Additional Experiment: Original Features Only + MinMaxScaler + SMOTE + Random Forest

This experiment is performed to investigate whether the domain-based feature engineering
introduced in ML-2 contributes to the model performance.

Unlike the four main experiments, this pipeline uses only the original six predictive
features from the dataset:

- Type
- Air temperature [K]
- Process temperature [K]
- Rotational speed [rpm]
- Torque [Nm]
- Tool wear [min]

The pipeline applies:

1. One-Hot Encoding for the categorical `Type` feature
2. MinMaxScaler for numerical features
3. SMOTE for class-imbalance handling
4. Random Forest classifier

The experiment is evaluated using the same 5-fold Stratified Cross-Validation strategy
and the same evaluation metrics used in ML-3.

This is an additional diagnostic experiment and does not replace the selected
ML-3 winning pipeline.

In [9]:
# ============================================================
# ADDITIONAL EXPERIMENT
# Original Features + OneHot + MinMaxScaler + SMOTE + RF
# ============================================================

# Original dataset features only
original_categorical_cols = ['Type']

original_numerical_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

# Preprocessor using ONLY the original features
original_feature_preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                sparse_output=False,
                handle_unknown='ignore'
            ),
            original_categorical_cols
        ),
        (
            'num',
            MinMaxScaler(),
            original_numerical_cols
        )
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# No DomainFeatureEngineer() here.
# Therefore, no engineered features are created.

original_smote_rf_pipeline = ImbPipeline(
    steps=[
        ('preprocessor', original_feature_preprocessor),

        ('smote', SMOTE(random_state=42)),

        (
            'classifier',
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# Same 5-fold Stratified CV used in ML-3
original_cv_results = cross_validate(
    original_smote_rf_pipeline,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    n_jobs=-1
)

original_experiment_result = {
    "Experiment": "Original Features + OneHot + MinMaxScaler + SMOTE + RF",
    "CV Recall": round(np.mean(original_cv_results['test_recall']), 4),
    "CV Precision": round(np.mean(original_cv_results['test_precision']), 4),
    "CV F1": round(np.mean(original_cv_results['test_f1']), 4),
    "CV PR-AUC": round(np.mean(original_cv_results['test_pr_auc']), 4),
    "CV ROC-AUC": round(np.mean(original_cv_results['test_roc_auc']), 4)
}

print("=" * 80)
print("ADDITIONAL EXPERIMENT RESULT")
print("=" * 80)

print(
    pd.DataFrame([original_experiment_result]).to_string(index=False)
)

print("\nThis experiment uses ORIGINAL FEATURES ONLY.")
print("No domain-based feature engineering is applied.")

ADDITIONAL EXPERIMENT RESULT
                                            Experiment  CV Recall  CV Precision  CV F1  CV PR-AUC  CV ROC-AUC
Original Features + OneHot + MinMaxScaler + SMOTE + RF     0.7491        0.4801 0.5845     0.6802      0.9617

This experiment uses ORIGINAL FEATURES ONLY.
No domain-based feature engineering is applied.
